# NYC Yellow Taxi — 2025 Full-Year Data Pipeline

This notebook applies a validated data-cleaning and feature-engineering pipeline
to the 2025 NYC Yellow Taxi trip data.

The cleaned monthly datasets will be used for downstream SQL Server analysis
and Power BI reporting.

## 1. Setup

In [5]:
import pandas as pd
import numpy as np
import os

## 2. Identify Monthly Source Files

The 2025 Yellow Taxi data is provided as one Parquet file per month.
The files are identified programmatically so the same processing pipeline
can be applied consistently across the full year.

In [6]:
parquet_files = sorted(
    file for file in os.listdir("data")
    if file.endswith(".parquet"))

## 3. Data Cleaning & Feature Engineering

The cleaning rules were established and validated during the January
dataset analysis and are applied consistently across all 2025 months.

### Cleaning rules
- Keep trips with positive duration
- Remove trips longer than 180 minutes
- Remove trips with distance above 100 miles
- Remove records with negative fares

### Features created
- Trip duration in minutes
- Pickup date
- Pickup hour
- Day of week
- Weekday / weekend classification
- Average speed in mph

Unrealistic calculated speeds are treated as missing rather than
being used as valid observations.

In [8]:
def clean_taxi_data(df):

    df = df.copy()

    # Calculate trip duration in minutes
    df["trip_duration_minutes"] = (
        df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    # Remove records unsuitable for trip-level mobility analysis
    df = df[
    (df["tpep_pickup_datetime"] >= "2025-01-01") &
    (df["tpep_pickup_datetime"] < "2026-01-01") &
    (df["trip_duration_minutes"] > 0) &
    (df["trip_duration_minutes"] <= 180) &
    (df["trip_distance"] <= 100) &
    (df["fare_amount"] >= 0)].copy()

    # Create time features
    df["pickup_date"] = df["tpep_pickup_datetime"].dt.date
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
    df["day_name"] = df["tpep_pickup_datetime"].dt.day_name()

    df["day_type"] = np.where(
        df["tpep_pickup_datetime"].dt.dayofweek < 5,
        "Weekday",
        "Weekend"
    )

    # Calculate average speed
    df["avg_speed_mph"] = (
        df["trip_distance"] /
        (df["trip_duration_minutes"] / 60)
    )

    # Treat unrealistic speeds as unavailable
    df.loc[
        ~df["avg_speed_mph"].between(0, 80),
        "avg_speed_mph"
    ] = np.nan

    # Keep only fields required for the project
    df = df[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_distance",
            "PULocationID",
            "DOLocationID",
            "fare_amount",
            "total_amount",
            "trip_duration_minutes",
            "pickup_date",
            "pickup_hour",
            "day_name",
            "day_type",
            "avg_speed_mph"
        ]
    ]

    return df

## 4. Process Monthly Data

Each monthly dataset is cleaned independently and saved as a Parquet file.
Processing the months separately avoids loading the entire year's dataset
into memory at once and creates reusable monthly outputs for downstream analysis.

In [9]:
cleaned_folder = "cleaned_data"
os.makedirs(cleaned_folder, exist_ok=True)

for file in parquet_files:

    month = file.replace(
        "yellow_tripdata_2025-", ""
    ).replace(".parquet", "")

    print(f"Processing {month}...")

    df = pd.read_parquet(
        os.path.join("data", file)
    )

    df_clean = clean_taxi_data(df)

    output_path = os.path.join(
        cleaned_folder,
        f"yellow_tripdata_2025-{month}_clean.parquet"
    )

    df_clean.to_parquet(
        output_path,
        index=False
    )

    print(
        f"  Rows after cleaning: {len(df_clean):,}"
    )

Processing 01...
  Rows after cleaning: 3,327,554
Processing 02...
  Rows after cleaning: 3,388,311
Processing 03...
  Rows after cleaning: 3,912,533
Processing 04...
  Rows after cleaning: 3,747,243
Processing 05...
  Rows after cleaning: 4,200,531
Processing 06...
  Rows after cleaning: 3,977,288
Processing 07...
  Rows after cleaning: 3,594,432
Processing 08...
  Rows after cleaning: 3,263,663
Processing 09...
  Rows after cleaning: 3,939,423
Processing 10...
  Rows after cleaning: 4,037,616
Processing 11...
  Rows after cleaning: 3,722,491
Processing 12...
  Rows after cleaning: 4,196,482


## 5. Validate Cleaned Outputs

Confirm that a cleaned Parquet file was created for each month of 2025.

In [10]:
cleaned_files = sorted(
    os.listdir(cleaned_folder))

print(f"Cleaned files created: {len(cleaned_files)}")

cleaned_files

Cleaned files created: 13


['data',
 'yellow_tripdata_2025-01_clean.parquet',
 'yellow_tripdata_2025-02_clean.parquet',
 'yellow_tripdata_2025-03_clean.parquet',
 'yellow_tripdata_2025-04_clean.parquet',
 'yellow_tripdata_2025-05_clean.parquet',
 'yellow_tripdata_2025-06_clean.parquet',
 'yellow_tripdata_2025-07_clean.parquet',
 'yellow_tripdata_2025-08_clean.parquet',
 'yellow_tripdata_2025-09_clean.parquet',
 'yellow_tripdata_2025-10_clean.parquet',
 'yellow_tripdata_2025-11_clean.parquet',
 'yellow_tripdata_2025-12_clean.parquet']